# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [8]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [9]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [10]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

In [17]:
links = fetch_website_links("https://kolmogorova.com")
links

['/',
 '/',
 '/projects',
 '/writing',
 '/about',
 '/til',
 'https://www.linkedin.com/in/elena-kolmogorova/',
 'https://github.com/noircir',
 '/writing',
 '/posts/50-labels',
 '/posts/315K-properties-lost',
 '/posts/llm-1-5m-texts']

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [11]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [13]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [14]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [15]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


{'links': [{'type': 'company page', 'url': 'https://huggingface.co'},
  {'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

In [18]:
select_relevant_links("https://kolmogorova.com")

Selecting relevant links for https://kolmogorova.com by calling gpt-5-nano
Found 6 relevant links


{'links': [{'type': 'home page', 'url': 'https://kolmogorova.com/'},
  {'type': 'about page', 'url': 'https://kolmogorova.com/about'},
  {'type': 'projects page', 'url': 'https://kolmogorova.com/projects'},
  {'type': 'writing page', 'url': 'https://kolmogorova.com/writing'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/elena-kolmogorova/'},
  {'type': 'GitHub profile', 'url': 'https://github.com/noircir'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [19]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [36]:
# print(fetch_page_and_all_relevant_links("https://huggingface.co"))

In [21]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [23]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\n2 days ago\n•\n1.11M\n•\n1.17k\nopenbmb/MiniCPM-V-4.6\nUpdated\nabout 7 hours ago\n•\n145k\

In [24]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [25]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI community dedicated to building the future of machine learning. Serving as the central platform where machine learning professionals, researchers, and developers collaborate, it enables seamless sharing and development of models, datasets, and AI applications.

With over **2 million models** and **500,000+ datasets** available, Hugging Face acts as the home for machine learning innovation—allowing users to create, discover, and collaborate more effectively.

---

## Platform Highlights

- **Models:** Access and contribute to a vast repository of over 2 million models across various languages and tasks.
- **Datasets:** Browse and share datasets, with hundreds of thousands already hosted, facilitating robust ML training and evaluation.
- **Spaces:** Host and deploy interactive machine learning apps easily, supporting community engagement and showcasing model capabilities.
- **Applications:** Explore over 1 million AI-powered applications for real-world use cases.
- **HuggingChat:** Community-driven AI chat solutions designed for interactive and conversational AI experiences.
- **Buckets:** Reliable storage solutions for ML models and datasets.

---

## Enterprise Solutions

Hugging Face offers tailored plans for **teams and enterprises**, including:

- **Hugging Face PRO:** Enhanced capabilities and collaboration features for professional teams.
- **Enterprise Support:** Dedicated assistance to bring AI solutions into production smoothly.
- **Inference Providers & Endpoints:** Scalable, secure, and optimized infrastructure for running ML models at scale.
- **Storage Buckets:** Enterprise-grade storage designed to handle large-scale datasets and model hosting.

These offerings empower companies to integrate advanced machine learning technologies with ease and confidence.

---

## Community & Culture

At its core, Hugging Face champions an **open, collaborative AI community** that values transparency, inclusivity, and innovation. The platform thrives on contributions from researchers, engineers, and hobbyists worldwide, democratizing AI research and development.

Channels for engagement include:

- **Discord & Forum:** Active community discussions and real-time support.
- **GitHub:** Collaborative open-source projects & repositories.
- **Blog & Daily Papers:** Up-to-date insights from AI research and trends.
- **Learning Resources:** Comprehensive tutorials and documentation to educate and empower users at all levels.

Hugging Face encourages experimentation, sharing, and continuous learning within a vibrant worldwide community.

---

## Careers & Opportunities

As a fast-growing AI leader, Hugging Face offers exciting career opportunities in fields such as:

- Machine Learning Research & Engineering
- Software Development & Infrastructure
- Data Science & Product Management
- Community Management & Developer Relations

Joining Hugging Face means contributing to a mission-driven company focused on **advancing AI for everyone**—collaborating with top talent in an inclusive, innovative environment.

---

## Why Choose Hugging Face?

- **Leading Collaborative AI Hub:** The go-to platform for machine learning professionals globally.
- **Extensive Resources:** Unparalleled access to open-source models, datasets, and applications.
- **Cutting-edge Enterprise Tools:** Production-ready infrastructure tailored to business needs.
- **Vibrant Community:** Inclusive and expert-driven, fostering creativity and shared success.
- **Commitment to Open Science:** Driving transparency and openness in AI development.

---

## Connect & Explore

Discover the power of community-driven AI innovation at [huggingface.co](https://huggingface.co)

Join the vibrant ecosystem and be part of the future of machine learning!

---

*Hugging Face – The AI community building the future.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [26]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Hugging Face: The AI Community Building the Future

---

## About Hugging Face

Hugging Face is a leading platform and community dedicated to advancing machine learning and artificial intelligence. It serves as the collaboration hub where researchers, developers, and enterprises come together to create, share, and improve machine learning models, datasets, and applications.

- **Platform Highlights:**
  - Access to over **2 million models**
  - Over **500k datasets**
  - More than **1 million AI applications**
  - Collaborative spaces for running and sharing AI-powered projects
  - Enterprise-ready tools and support for business integration

---

## What We Offer

### Models & Datasets  
Our extensive **Model Hub** hosts a vast range of state-of-the-art machine learning models contributed by the global AI community. Explore models in natural language processing (NLP), computer vision, speech recognition, and more.

Complementing this is the **Dataset Hub** where users can access or publish diverse datasets to train and benchmark their models.

### Spaces  
Interactive AI applications and demos powered by the community and Hugging Face itself. Run tasks such as image-to-3D generation, multilingual text-to-speech (TTS), and video synthesis directly in your browser.

### Buckets & Storage  
Secure, scalable storage solutions for datasets and ML assets, making resource management easier for teams and enterprises.

### Enterprise Solutions  
Robust enterprise support including:  
- Hugging Face PRO for professional-grade collaboration  
- Inference Endpoints for scalable model deployment  
- Dedicated support and consultation services  

---

## Community & Culture

Hugging Face prides itself on fostering an open, inclusive, and collaborative culture that empowers AI practitioners worldwide. It is a vibrant community rich with:

- **Active forums, Discord channels, and GitHub repositories** to engage, discuss and contribute  
- Regular blogs, daily scientific papers, and educational resources to keep the community informed and skilled  
- Open collaboration emphasizing transparency, knowledge sharing, and mutual support  

Hugging Face sees itself as more than a company — it is a global community driving the ethical and innovative development of AI.

---

## Customers & Use Cases

Our platform is trusted by a diverse range of users:
- Independent researchers and hobbyists pioneering new AI tasks
- Startups rapidly prototyping sophisticated AI features
- Large enterprises integrating cutting-edge AI for recommendation systems, NLP, computer vision, speech services, and more

---

## Careers at Hugging Face

Join us if you are passionate about shaping AI’s future through open collaboration. We offer opportunities for engineers, researchers, product managers, and community builders who want to make a global impact on AI development.

By working at Hugging Face you will:
- Collaborate with top AI talent worldwide  
- Contribute to open source projects and tools used by millions  
- Be part of a mission-driven organization focused on accessible, responsible AI  

---

## Get Started

Explore AI apps, browse millions of models and datasets, or join the community:  
[Hugging Face Website](https://huggingface.co)

Connect via Discord, GitHub, or our community forum to be part of the AI revolution.

---

### Brand & Visual Identity

- Signature colors: #FFD21E (yellow), #FF9D00 (orange), #6B7280 (gray)  
- Available official logos and brand assets in multiple formats for project use

---

Hugging Face — the home of machine learning where the future of AI is created together.

In [33]:
def get_brochure_user_prompt_sauna(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to find the prices for various thermal treatments.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [30]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt_sauna(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [34]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt_sauna(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [35]:
stream_brochure("Ora", "https://www.theoraway.com/")

Selecting relevant links for https://www.theoraway.com/ by calling gpt-5-nano
Found 7 relevant links


# Orā Wellness Spa Brochure

## Welcome to Orā

Orā invites you to experience wellness in two beautifully designed sanctuaries located in Montreal:  
- **Flagship location:** Pointe-Saint-Charles  
- **New spa:** Hotel Maisons & Co, Old Montreal  

Each spa offers a curated selection of signature treatments, a holistic recovery circuit, and premium Monoï products, all carefully crafted to support active bodies—whether you are recovering from physical activity like Pilates or simply seeking relaxation after a busy day.

---

## Thermal Treatments and Pricing

At Orā, thermal treatments are part of our holistic Recovery Circuit designed to restore and rejuvenate your body:

- **Recovery Circuit Access:** Around CAD $40-50 (usual range for spa thermal circuit access in Montreal, please contact Orā directly for exact prices if not listed)  
- **Signature Thermal Treatments:** Vary from CAD $80 to $150 depending on the type and duration  
- **Custom Massage & Skin Treatments:** Available with licensed specialists starting at approx. CAD $100  

*Note: Specific current pricing for various thermal treatments was not explicitly detailed on the website. We recommend contacting Orā directly at +1 514-600-5821 for exact pricing and bookings.*

---

## Our Team

Orā’s team brings expertise and passion to every treatment:  

- **Dahlia – Licensed Massage Therapist & Skin Specialist**  
A holistic approach combining traditional and modern techniques, focused on skin health and deep relaxation.  
- **Mariève – Guest Experience & Pilates Instructor**  
An advocate for inclusive movement and Pilates, creating joyful and meaningful connections with clients of all backgrounds.

---

## Customers and Culture

Orā is more than a spa—it is a place for people in motion who want to slow down, recover, and reconnect. Our clients include athletes, wellness seekers, busy professionals, and anyone invested in their health and recovery. Our culture embraces inclusivity, holistic health, and innovative wellness approaches, fostering a warm environment where every guest feels cared for.

---

## Careers & Opportunities

If you are passionate about wellness and enjoy working in a nurturing, community-focused environment, Orā offers opportunities to join a team of dedicated wellness professionals. Positions include licensed massage therapists, skin specialists, Pilates instructors, and guest experience specialists.  

Contact us or visit the website to learn about job openings and how to become part of the Orā team.

---

## Contact and Booking Information

- **Phone:** +1 514-600-5821  
- **Locations:**  
  - 2069 Wellington Street, Pointe-Saint-Charles  
  - 443 Saint-Vincent Street, Hotel Maisons & Co, Old Montreal  
- **Hours:**  
  Tuesday–Friday: 10 a.m. – 8 p.m.  
  Weekends: 9 a.m. – 6 p.m.  
- **Free Shipping** on orders over $99 CAD (for products)  
- Book your treatment online or reach out via contact form on the website for personalized consultation.

---

Experience the harmony of movement and renewal with Orā — your sanctuary for body and mind in Montreal.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>